# AML Alert Prioritization | DnkCode

Train-only exploratory analysis and baseline modeling for WIUT Hackathon 2026. Run from the repository root after installing `requirements.txt`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / 'fintech_data'
signals = pd.read_csv(DATA / 'train_signals.csv')
signals['signal_sanasi'] = pd.to_datetime(signals['signal_sanasi'])
signals.head()

In [ ]:
signals['eskalatsiya'].value_counts(normalize=True).rename(index={0: 'dismissed', 1: 'escalated'})

In [ ]:
ax = signals['eskalatsiya'].value_counts().sort_index().plot.bar(color=['#4a8f79', '#d36a52'])
ax.set_xticklabels(['Dismissed', 'Escalated'], rotation=0)
ax.set_ylabel('Alerts')
ax.set_title('Training target distribution');

## Initial observations

Training labels contain 2,405 escalations out of 14,000 alerts (17.18%), so the target is imbalanced. The transaction table has 6,987,663 rows: 75.6% are incoming; card transactions make up 53.8% and bank transfers 39.4%. Median transaction count is 493 for escalated alerts and 455 for dismissed alerts. This difference motivates alert-level activity features, but does not establish causation or guarantee predictive value.

## Transaction behavior

The 6.99M-row transaction table is summarized by alert in `src/model.py`. These plots characterize the overall training transaction history; outcome comparisons should be interpreted as descriptive, not causal.

In [ ]:
tx = pd.read_parquet(DATA / 'train_transactions.parquet')
tx['tranzaksiya_vaqti'] = pd.to_datetime(tx['tranzaksiya_vaqti'], errors='coerce')
tx.shape, tx['signal_id'].nunique(), tx.isna().mean()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
tx.set_index('tranzaksiya_vaqti').resample('W').size().plot(ax=axes[0], title='Weekly transaction count')
tx['kirim_chiqim'].value_counts().plot.bar(ax=axes[1], title='Direction')
tx['tranzaksiya_turi'].value_counts().plot.bar(ax=axes[2], title='Transaction type')
plt.tight_layout()

In [ ]:
labeled_tx = tx.merge(signals[['signal_id', 'eskalatsiya']], on='signal_id', how='inner')
labeled_tx.groupby('eskalatsiya')['miqdor_indeksi'].agg(['count', 'median', 'mean', 'std'])

## Modeling decisions and current limitation

The target is imbalanced, so the baseline uses stratified holdout validation and class-weighted logistic regression; ROC-AUC evaluates ranking quality. Features include transaction count, amount summaries, direction/type counts, history duration, and alert date parts. `test_transactions.parquet` in the supplied folder is corrupt and cannot be read, so a final test prediction file cannot be generated until a clean copy is obtained.